***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)
    * 下一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.6 互相关、自相关与相似性度量<a id='math:sec:cross_correlation_and_auto_correlation'></a>

卷积描述“输入经过系统响应后变成什么”；相关描述“两个信号在某个相对延迟或相对位移下有多相似”。射电干涉测量的核心硬件操作正是相关：两个天线接收到的电场先被延迟补偿和下变频，然后相关器计算它们的复共轭乘积并积分，得到一条基线上的复可见度。

因此，本节不是信号处理中的附属小节，而是连接第 2.1 节相量共轭乘积与后续可见度测量方程的桥梁。互相关用于比较两个不同信号，自相关用于比较信号与其自身；二者都可以作为函数，也可以作为随机过程的统计平均。


### 2.6.1 互相关：寻找相对延迟<a id='math:sec:cross_correlation'></a>

从最直观的情形开始：两个信号形状相似，但其中一个相对另一个延迟了。若把一个信号沿时间轴平移，并在每个平移量上计算两者的重叠程度，那么重叠最大的位置就给出了相对延迟。

![互相关峰值给出相对延迟](figures/correlation_delay_peak.png)

**图 2.6.1** 两个相似波包及其互相关。互相关函数的峰值出现在两者最匹配的相对延迟附近。


对复值函数，本书采用以下互相关约定：

<a id='math:eq:cross_correlation_definition'></a><!--\label{math:eq:cross_correlation_definition}-->
$$
(f\star g)(\tau)=\int_{-\infty}^{+\infty}f^*(t)g(t+\tau)\,dt.
$$

其中 $\tau$ 是延迟或位移。这个定义可以看成“把 $g$ 平移 $\tau$，再与 $f$ 做复内积”。复共轭不可省略，因为它使相关成为复向量空间中的内积形式，也使相位差以正确的符号进入结果。若 $f$ 与 $g$ 都是实函数，复共轭当然不起作用。

互相关也可以写成卷积形式。令 $f_-^*(t)=f^*(-t)$，则

<a id='math:eq:correlation_as_convolution'></a><!--\label{math:eq:correlation_as_convolution}-->
$$
(f\star g)(\tau)=(f_-^* * g)(\tau).
$$

这说明相关与卷积非常接近，但相关的目的不是描述系统输出，而是度量相似性。对非对称函数而言，是否反向、是否取共轭会改变结果，因此二者不能混用。


互相关满足一个重要对称关系：

<a id='math:eq:cross_correlation_symmetry'></a><!--\label{math:eq:cross_correlation_symmetry}-->
$$
(g\star f)(\tau)=(f\star g)^*(-\tau).
$$

证明直接来自定义：

$$
\begin{aligned}
(g\star f)(\tau)
&=\int g^*(t)f(t+\tau)dt\\
&=\left[\int g(t)f^*(t+\tau)dt\right]^*.
\end{aligned}
$$

令 $u=t+\tau$，便得到右侧的反号关系。对实信号，这个式子退化为普通的镜像对称；对复信号，共轭体现了相位方向的反转。


### 2.6.2 互相关与干涉相关器<a id='math:sec:correlator_visibility'></a>

设两台天线接收到的窄带复电场为 $E_p(t)$ 与 $E_q(t)$。一个理想化的相关器会形成类似

<a id='math:eq:visibility_correlation_skeleton'></a><!--\label{math:eq:visibility_correlation_skeleton}-->
$$
R_{pq}(\tau)=\left\langle E_p(t)E_q^*(t+\tau)\right\rangle_t
$$

的量，其中尖括号表示在有限时间内平均。不同教材和软件可能把共轭放在第一个或第二个信号上，也可能使用相反的延迟符号；关键是要在同一套约定内保持一致。

若两个信号只相差一个相位，

$$
E_p(t)=A_pe^{i\omega t},
\qquad
E_q(t)=A_qe^{i(\omega t+\phi)},
$$

则零延迟相关为

$$
\left\langle E_p(t)E_q^*(t)\right\rangle_t
=A_pA_qe^{-i\phi}.
$$

快速振荡的 $e^{i\omega t}$ 被共轭乘积消去，留下稳定的相对相位。经过几何延迟补偿、带宽平均和校准之后，这类复相关量就成为可见度的观测基础。


### 2.6.3 归一化相关与相干性<a id='math:sec:normalized_correlation'></a>

互相关的数值大小同时受信号本身强度影响。为了只度量相似程度，常定义归一化相关系数

<a id='math:eq:normalized_correlation'></a><!--\label{math:eq:normalized_correlation}-->
$$
\rho_{fg}(\tau)=
\frac{(f\star g)(\tau)}
{\left[\int |f(t)|^2dt\right]^{1/2}
 \left[\int |g(t)|^2dt\right]^{1/2}}.
$$

由 Cauchy-Schwarz 不等式可知 $|\rho_{fg}(\tau)|\le 1$。当两个信号在某个延迟下只差一个复常数因子时，归一化相关的模达到 1；当二者不相关时，长时间平均后相关趋近于 0。射电干涉测量中的相干性、相位稳定性和信噪比判断，都与这种归一化思想有关。


### 2.6.4 自相关：信号与自身的相似性<a id='math:sec:auto_correlation'></a>

自相关是互相关的特例：

<a id='math:eq:auto_correlation_definition'></a><!--\label{math:eq:auto_correlation_definition}-->
$$
R_f(\tau)=(f\star f)(\tau)=\int_{-\infty}^{+\infty}f^*(t)f(t+\tau)\,dt.
$$

它回答的问题是：信号与自身平移 $\tau$ 后仍有多相似。零延迟处给出信号能量：

<a id='math:eq:auto_correlation_zero_lag'></a><!--\label{math:eq:auto_correlation_zero_lag}-->
$$
R_f(0)=\int |f(t)|^2dt.
$$

并且

<a id='math:eq:auto_correlation_symmetry'></a><!--\label{math:eq:auto_correlation_symmetry}-->
$$
R_f(-\tau)=R_f^*(\tau).
$$

对实信号，自相关是偶函数；对复信号，它具有 Hermitian 对称性。


自相关可以揭示周期性和相关长度。若信号中有稳定周期，自相关会在相应的延迟处反复出现峰值；若信号只有短程相关，自相关会很快衰减。对随机过程，常用时间平均或集合平均定义自相关函数，它反映的是统计意义上的相似性，而不是某一次有限观测中的精确积分。

![自相关与功率谱](figures/correlation_autocorr_power.png)

**图 2.6.2** 自相关与功率谱。周期成分会在自相关函数中产生重复峰，也会在功率谱中产生频率峰。二者之间的严格联系由维纳-辛钦定理给出。


### 2.6.5 维纳-辛钦关系的预告<a id='math:sec:wiener_khinchin_preview'></a>

自相关和功率谱并不是两种独立描述。对适当的信号或平稳随机过程，自相关函数的傅里叶变换等于功率谱密度：

<a id='math:eq:wiener_khinchin_preview'></a><!--\label{math:eq:wiener_khinchin_preview}-->
$$
\mathcal{F}\{R_f\}(s)=|F(s)|^2.
$$

这个关系称为维纳-辛钦定理。它解释了为什么自相关会在频谱估计、噪声分析和接收机功率测量中反复出现。后面讨论傅里叶定理和观测噪声时，会更系统地使用这一思想。


### 2.6.6 本节要点

互相关度量两个信号在不同延迟下的相似程度；自相关度量一个信号与自身的相似程度。复共轭使相关同时携带幅度与相位信息，这正是干涉相关器能够从两个天线的电场中提取复可见度的数学原因。卷积与相关形式相近，但物理问题不同：卷积描述系统响应如何重塑输入，相关描述两个信号如何匹配。


***

* 下一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
